In [6]:
from scipy.stats import entropy
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, average_precision_score, accuracy_score
from sklearn.preprocessing import label_binarize
import gc
import joblib
import time
import lightgbm as lgb
import pandas as pd
import torch
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

filenames = {
    "Original":         "cleaned_original_data.csv",
    "Small GAN":        "small_GAN_combined.csv",
    "Med GAN":          "med_GAN_combined.csv",
    "Large GAN":        "large_GAN_combined.csv",
    "Small Diffusion":  "small_Diffusion_combined.csv",
    "Med Diffusion":    "med_Diffusion_combined.csv",
    "Large Diffusion":  "large_Diffusion_combined.csv",
    "Small LLM":        "small_LLM_combined.csv",
    "Med LLM":          "med_LLM_combined.csv",
    # "Large LLM":        "large_LLM_combined.csv",
}

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

target = 'class1'
drop_cols = ['class2', 'class3']

df_orig = pd.read_csv(filenames["Original"])
X_all_orig = df_orig.drop(columns=[target] + drop_cols, errors='ignore')
y_all_orig = df_orig[target]

X_train_orig, X_test_orig, y_train_orig, y_test_orig = train_test_split(
    X_all_orig, y_all_orig, test_size=0.2, random_state=42
)
all_possible_labels = sorted(y_all_orig.unique())

orig_dist = (pd.Series(y_train_orig).value_counts(normalize=True)
             .reindex(all_possible_labels, fill_value=0)
             .sort_index().values)

num_classes = y_all_orig.nunique()
encoders = joblib.load('label_encoders.joblib')
le = encoders['class1']
results = []
dist_data = []

for name, path in filenames.items():
    print(f"Processing {name}...")
    start_time = time.time()

    if name == "Original":
        X_train, y_train = X_train_orig, y_train_orig
    else:
        df_temp = pd.read_csv(path)
        y_train_raw = pd.Series(df_temp[target]).astype(
            pd.CategoricalDtype(categories=all_possible_labels))
        X_train_raw = df_temp.reindex(columns=X_test_orig.columns)
        mask = y_train_raw.notna()
        if not mask.all():
            invalid_count = (~mask).sum()
            print(
                f"⚠️ Warning: {name} contained {invalid_count} unrecognized labels. Dropping those rows.")
            y_train = y_train_raw[mask]
            X_train = X_train_raw[mask]
        else:
            y_train = y_train_raw
            X_train = X_train_raw
        X_train = X_train.fillna(0)
        del df_temp

    current_dist = (y_train.value_counts(normalize=True)
                    .reindex(all_possible_labels, fill_value=0)
                    .sort_index().values)
    kl_div = entropy(orig_dist, current_dist + 1e-10)
    counts = y_train.value_counts(normalize=True)
    for label_idx, ratio in counts.items():
        # Map the integer back to the string name
        class_name = le.inverse_transform([int(label_idx)])[0]

        dist_data.append({
            "Dataset": name,
            "Class": class_name,
            "Percentage": ratio * 100
        })

    model = lgb.LGBMClassifier(
        min_data_in_leaf=1,
        min_sum_hessian_in_leaf=0.001,
        zero_as_missing=True,
        objective='multiclass',
        num_class=num_classes,
        device='cpu',
        gpu_platform_id=0,
        gpu_device_id=0,
        n_estimators=200,
        learning_rate=0.05,
        verbosity=-1
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test_orig)
    y_prob_raw = model.predict_proba(X_test_orig)
    y_prob_full = np.zeros((len(X_test_orig), num_classes))
    classes_seen = model.classes_

    for i, class_val in enumerate(classes_seen):
        col_idx = all_possible_labels.index(class_val)
        y_prob_full[:, col_idx] = y_prob_raw[:, i]

    results.append({
        "Dataset": name,
        "KL Divergence": round(kl_div, 4),
        "Accuracy": round(accuracy_score(y_test_orig, y_pred), 4),
        "F1 (Macro)": round(f1_score(y_test_orig, y_pred, average='macro'), 4),
        "PR-AUC (Macro)": round(average_precision_score(label_binarize(y_test_orig, classes=all_possible_labels), y_prob_full, average='macro'), 4)
    })

    if name == "Original":
        del X_train, y_train
        del X_train_orig, y_train_orig
    else:
        del X_train, y_train

    gc.collect()
df_res = pd.DataFrame(results)

Processing Original...
Processing Small GAN...
Processing Med GAN...
Processing Large GAN...
Processing Small Diffusion...
Processing Med Diffusion...
Processing Large Diffusion...
Processing Small LLM...
Processing Med LLM...


In [ ]:
print(df_res)
print(dist_data)


def split_name(name):
    if name == "Original":
        return "Original", "Original"
    parts = name.split(" ")
    return parts[0], parts[1]


df_res[['Size', 'Method']] = df_res['Dataset'].apply(
    lambda x: pd.Series(split_name(x)))

df_res = df_res[['Method', 'Size', 'Accuracy',
                 'F1 (Macro)', 'PR-AUC (Macro)', 'KL Divergence']]

print(df_res)
df_res.to_csv('pivoted_results_full.csv', index=False)

df_dist = pd.DataFrame(dist_data)
df_dist['Proportion'] = df_dist['Percentage'] / 100


orig_df = df_dist[df_dist['Dataset'] == 'Original'].set_index('Class')
original_dist = orig_df['Proportion']
dataset_names = list(filenames.keys())


threshold = 0.01  # 1%
common_classes = original_dist[original_dist > threshold].index.tolist()
rare_classes = original_dist[original_dist <= threshold].index.tolist()


df_runtimes = pd.read_csv('runtimes_log.csv')
df_runtimes[['Size', 'Method']] = df_runtimes['Model'].str.split(
    ' ', expand=True)

size_order = ['Small', 'Med', 'Large']
method_order = ['GAN', 'Diffusion', 'LLM']


df_runtimes['Size'] = pd.Categorical(
    df_runtimes['Size'], categories=size_order, ordered=True)
df_runtimes['Method'] = pd.Categorical(
    df_runtimes['Method'], categories=method_order, ordered=True)
df_runtimes = df_runtimes.sort_values(['Method', 'Size'])

           Dataset  KL Divergence  Accuracy  F1 (Macro)  PR-AUC (Macro)
0         Original         0.0000    0.5756      0.1571          0.1003
1        Small GAN         0.2214    0.9495      0.5277          0.5275
2          Med GAN         0.0008    0.7559      0.2651          0.2013
3        Large GAN         0.0006    0.7249      0.3674          0.2614
4  Small Diffusion         0.2214    0.7789      0.2362          0.3026
5    Med Diffusion         0.2625    0.8277      0.3250          0.3999
6  Large Diffusion         0.2717    0.7145      0.2236          0.2222
7        Small LLM         0.2214    0.9496      0.5277          0.5275
8          Med LLM         0.0008    0.5196      0.0467          0.0597
[{'Dataset': 'Original', 'Class': 'Normal', 'Percentage': 51.33713130094858}, {'Dataset': 'Original', 'Class': 'RDOS', 'Percentage': 17.224102931927447}, {'Dataset': 'Original', 'Class': 'Scanning_vulnerability', 'Percentage': 6.434311454664236}, {'Dataset': 'Original', 'Class': 

In [27]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

color_map = {
    'Small': '#1f77b4',  # Muted Blue
    'Med': '#ff7f0e',    # Safety Orange
    'Large': '#2ca02c'   # Cooked Asparagus Green
}

metrics = ['Accuracy', 'F1 (Macro)', 'PR-AUC (Macro)', 'KL Divergence']
metric_map = {m: m if m == 'Accuracy' else m.split(' ')[0] for m in metrics}

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[metric_map[m] for m in metrics],
    horizontal_spacing=0.12,
    vertical_spacing=0.15
)

for i, metric in enumerate(metrics):
    row = i // 2 + 1
    col = i % 2 + 1

    df_metric = df_plot[['Method', 'Size', metric]].rename(
        columns={metric: 'Score'})

    # 2. Add bars with fixed colors based on 'Size'
    for size in ['Small', 'Med', 'Large']:
        df_size = df_metric[df_metric['Size'] == size]
        fig.add_trace(go.Bar(
            x=df_size['Method'],
            y=df_size['Score'],
            name=size,
            legendgroup=size,
            showlegend=(i == 0),
            marker=dict(color=color_map[size],
                        line=dict(width=0.5, color='black'))
        ), row=row, col=col)

    # 3. Add horizontal baseline with a single label
    # Using add_hline is more robust for subplots than manual shapes
    fig.add_hline(
        y=orig_metrics[metric],
        line_dash="dash",
        line_color="red",
        line_width=2,
        annotation_text="Original",  # Fixed typo "Ongimal"
        annotation_position="top left",
        annotation_font_size=10,
        annotation_font_color="red",
        row=row, col=col
    )

fig.update_layout(
    height=700,
    width=900,
    template='simple_white',
    title={'text': "Synthetic Data Quality Comparisons",
           'x': 0.5, 'xanchor': 'center'},
    font=dict(size=12, family='Times New Roman'),
    legend=dict(title_text='Dataset Size', orientation='h',
                yanchor='bottom', y=1.02, xanchor='right', x=1),
    barmode='group'
)

fig.write_image("synthetic_comparison.pdf", width=1200, height=600)
fig.show()

all_classes = sorted(df_dist['Class'].unique())

fig_dist = make_subplots(
    rows=2, cols=1,
    subplot_titles=(
        "Common Classes (>1%)",
        "Rare Classes (<1%)"
    ),
    vertical_spacing=0.12,
    row_heights=[0.4, 0.6]
)

fig_dist.update_layout(
    height=900,
    width=1100,
    barmode='group',
    template="plotly_white",
    title={
        'text': "Synthetic Data Class Distributions",
        'x': 0.5,
        'xanchor': 'center',
        'font': dict(size=20)
    },
    showlegend=True
)


def get_vals(cls_name):
    return [
        df_dist[(df_dist['Dataset'] == ds) & (
            df_dist['Class'] == cls_name)]['Proportion'].values[0]
        if not df_dist[(df_dist['Dataset'] == ds) & (df_dist['Class'] == cls_name)].empty else 0
        for ds in dataset_names
    ]


for cls in common_classes:
    fig_dist.add_trace(go.Bar(
        x=dataset_names, y=get_vals(cls), name=str(cls),
        legendgroup="common", legendgrouptitle_text="Common"
    ), row=1, col=1)


for cls in rare_classes:
    fig_dist.add_trace(go.Bar(
        x=dataset_names, y=get_vals(cls), name=str(cls),
        legendgroup="rare", legendgrouptitle_text="Rare Attacks"
    ), row=2, col=1)


fig_dist.update_yaxes(type="log", row=1, col=1, tickformat=".1%")
fig_dist.update_yaxes(type="log", row=2, col=1, tickformat=".2%")


fig_dist.update_layout(
    height=900,
    width=1100,
    barmode='group',
    template="plotly_white",
    title_text="Synthetic Data Class Distributions",
    showlegend=True
)

fig_dist.write_image("synth_class_distributions.pdf")
fig_dist.show()


def format_seconds(seconds):
    if pd.isna(seconds) or seconds == 0:
        return "-"
    h, m, s = int(seconds // 3600), int((seconds %
                                         3600) // 60), int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


actual_counts = {}
for name, path in filenames.items():
    try:
        actual_counts[name] = len(pd.read_csv(path, usecols=[0]))
    except Exception:
        actual_counts[name] = 0


perf_df = df_runtimes.groupby('Model')['Runtime'].mean().reset_index()
perf_df['Rows'] = perf_df['Model'].map(actual_counts)

perf_df[['Size', 'Method']] = perf_df['Model'].str.split(' ', expand=True)


def combine_metrics(row):
    time_str = format_seconds(row['Runtime'])
    rows_str = f"{int(row['Rows']):,}" if not pd.isna(row['Rows']) else "0"
    return f"{time_str} | {rows_str}"


perf_df['Cell_Value'] = perf_df.apply(combine_metrics, axis=1)

final_table = perf_df.pivot(
    index='Method', columns='Size', values='Cell_Value')


size_order = ['Small', 'Med', 'Large']
size_mapping = {
    'Small': 'Small (800)',
    'Med': 'Med (8k)',
    'Large': 'Large (80k)'
}


existing_cols = [c for c in size_order if c in final_table.columns]
final_table = final_table[existing_cols]


final_table.columns = [size_mapping[c] for c in final_table.columns]


method_order = ['GAN', 'Diffusion', 'LLM']
existing_rows = [r for r in method_order if r in final_table.index]
final_table = final_table.reindex(existing_rows)

print("\n--- Model Generation Summary (Time | Total Rows) ---")
print(final_table)


final_table.to_csv("performance_summary_grid.csv")


--- Model Generation Summary (Time | Total Rows) ---
              Small (800)          Med (8k)        Large (80k)
Method                                                        
GAN        00:00:15 | 624  00:04:05 | 7,711  00:09:17 | 78,065
Diffusion  00:00:28 | 800  00:00:40 | 8,000  00:01:33 | 80,000
LLM        04:03:54 | 624  14:41:21 | 7,711                NaN
